## AutoGrad
Autograd is a core component of PyTorch that provides automatic differentiation for tensor operations. It enables gradient computations, which is essential for training machine learning models using optimization models like gradient descent.

In [1]:
import torch
print(torch.__version__)

if torch.cuda.is_available():
  print(f"Running on GPU {torch.cuda.get_device_name(0)}")
else:
  print("Running on CPU")

2.11.0+cpu
Running on CPU


In [17]:
x = torch.tensor(4.0, requires_grad=True)
y = x**2
z = torch.sin(y)
print(f"X : {x}")
print(f"Y : {y}")
print(f"Z : {z}")

X : 4.0
Y : 16.0
Z : -0.2879033088684082


In [18]:
x

tensor(4., requires_grad=True)

In [19]:
y

tensor(16., grad_fn=<PowBackward0>)

In [20]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [21]:
## Now we have to calculate dz/dx so
## according to chain rule of derivative
## dz/dx = dz/dy * dy/dx

## But by using autograd of PyTorch we simply write
z.backward()   ## and then
x.grad   ## to get the value of dz/dx

tensor(-7.6613)

In [22]:
y.grad

/tmp/ipykernel_1276/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


#### Now Using it on Neural Netwoks

In [31]:
import torch

x = torch.tensor(6.7)    ## Input feature
y = torch.tensor(0.0)    ## True label(binary)

w = torch.tensor(1.0)    ## Weight
b = torch.tensor(0.0)    ## bias

In [32]:
## Binary cross entropy loss for scaler
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8
    prediction = torch.clamp(prediction, min=epsilon, max=1 - epsilon)

    return -(target * torch.log(prediction) +
             (1 - target) * torch.log(1 - prediction))

In [33]:
## Forward Pass
z = w * x + b
y_pred = torch.sigmoid(z)

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [34]:
loss

tensor(6.7012)

In [35]:
## Derivatives
# 1. dl/dy(pred) : Derivative of loss with respect to (y_pred)
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz : Predicyion(y_pred) with respect to z(sigmoid)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db : z with respect to w and b
dz_dw = x
dz_db = 1    # Bias contributes directly to z

dl_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dl_db = dloss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of loss w.r.t weight (dw) : {dl_dw}")
print(f"Manual Gradient of loss w.r.t bias (db) : {dl_db}")

Manual Gradient of loss w.r.t weight (dw) : 6.691762447357178
Manual Gradient of loss w.r.t bias (db) : 0.998770534992218


In [36]:
## Now using Autograd
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

z = w*x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

loss.backward()
w.grad, b.grad

(tensor(6.6918), tensor(0.9988))

### Examples of using Autograd on Vectors

In [43]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x**2).mean()

y.backward()
x.grad

tensor([0.6667, 1.3333, 2.0000])

In [44]:
## Clearing Grad
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

In [45]:
y.backward()
x.grad

tensor(4.)

In [46]:
## If I run the above block again
y = x ** 2
y.backward()
x.grad

tensor(8.)

In [47]:
## So this is the problem, the gradient will accumulate
x.grad.zero_

<function Tensor.zero_()>